In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Read and process data (same as before)
df = pd.read_excel('../resources/bioassay_data.xlsx')

if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])

numeric_cols = ['temp', 'RH', '24h dead_females', '24h total_females', 
                '24h_dead_males', '24h_total_males']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

if 'mort_females' in df.columns:
    if df['mort_females'].dtype == 'object':
        df['mort_females'] = df['mort_females'].astype(str).str.rstrip('%').astype(float) / 100
else:
    if '24h dead_females' in df.columns and '24h total_females' in df.columns:
        df['mort_females'] = df['24h dead_females'] / df['24h total_females']

df_clean = df[df['Insecticide'] != 'OPC'].dropna(subset=['species', 'mort_females'])

import re
# Define customizable parameters
# Define palette mapping, converting Plotly rgb → hex
def rgb_to_hex(rgb: str) -> str:
    nums = list(map(int, re.findall(r"\d+", rgb)))
    return "#{:02x}{:02x}{:02x}".format(*nums)

TAXON_PALETTE = [rgb_to_hex(c) for c in px.colors.qualitative.Vivid]
TAXON_COLORS = {
    "gambiae": TAXON_PALETTE[1],
    "coluzzii": TAXON_PALETTE[0],
    "arabiensis": TAXON_PALETTE[2],
    "merus": TAXON_PALETTE[3],
    "melas": TAXON_PALETTE[4],
    "quadriannulatus": TAXON_PALETTE[5],
    "fontenillei": TAXON_PALETTE[6],
    "gcx1": TAXON_PALETTE[7],
    "gcx2": TAXON_PALETTE[8],
    "gcx3": TAXON_PALETTE[9],
    "gcx4": TAXON_PALETTE[10],
    "unassigned": "black",
}

# Plot dimensions (easily customizable)
PLOT_WIDTH = 450
PLOT_HEIGHT = 500

# Font settings
FONT_SIZE = 14
TITLE_FONT_SIZE = 16
FONT_FAMILY = "Fira Sans"

print(f"Data loaded and filtered. Shape: {df_clean.shape}")

Data loaded and filtered. Shape: (16, 13)


In [5]:
def create_boxplot(width=PLOT_WIDTH, height=PLOT_HEIGHT):
    fig = px.box(
        df_clean, 
        x='species', 
        y='mort_females',
        color='species',
        color_discrete_map=TAXON_COLORS,
        # points='all',
        title="Female Mosquito Mortality by Species",
        labels={
            'mort_females': 'Mortality Rate',
            'species': 'Species'
        }
    )
    
    # # Add sample sizes positioned to the right of each box
    # sample_sizes = df_clean.groupby('species').size()
    
    # for i, (species, n) in enumerate(sample_sizes.items()):
    #     # Position text to the right of each species box
    #     fig.add_annotation(
    #         x=i + 0.35,  # Offset to the right
    #         y=df_clean[df_clean['species'] == species]['mort_females'].max() * 0.95,
    #         text=f"n={n}",
    #         showarrow=False,
    #         font=dict(size=FONT_SIZE-2, family=FONT_FAMILY),
    #         bgcolor="white",
    #         bordercolor="gray",
    #         borderwidth=1
    #     )
    
    fig.update_layout(
        width=width,
        height=height,
        template="plotly_white",
        yaxis_tickformat='.0%',
        font=dict(size=FONT_SIZE, family=FONT_FAMILY),
        title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY),
        showlegend=False
    )
    
    return fig

boxplot_fig = create_boxplot()
boxplot_fig.show()

In [6]:
def create_barplot(width=PLOT_WIDTH, height=PLOT_HEIGHT):
    # Calculate summary statistics
    summary = df_clean.groupby('species').agg({
        'mort_females': ['mean', 'std', 'count']
    }).round(3)
    
    summary.columns = ['mean', 'std', 'count']
    summary = summary.reset_index()
    summary['se'] = summary['std'] / np.sqrt(summary['count'])
    summary['sample_size'] = summary['count'].astype(int)
    
    fig = px.bar(
        summary,
        x='species',
        y='mean',
        color='species',
        color_discrete_map=TAXON_COLORS,
        error_y='se',
        title="Mean Female Mortality ± SE by Species",
        labels={
            'mean': 'Mean Mortality Rate',
            'species': 'Species'
        },
        text='sample_size'
    )
    
    fig.update_traces(texttemplate='n=%{text}', textposition="outside")
    
    fig.update_layout(
        width=width,
        height=height,
        template="plotly_white",
        yaxis_tickformat='.0%',
        showlegend=False,
        font=dict(size=FONT_SIZE, family=FONT_FAMILY),
        title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY)
    )
    
    return fig

bar_fig = create_barplot()
bar_fig.show()

In [7]:
def create_violinplot(width=PLOT_WIDTH, height=PLOT_HEIGHT, transparent_bg=False):
    fig = px.violin(
        df_clean,
        x="species",
        y="mort_females",
        color="species",
        color_discrete_map=TAXON_COLORS,
        box=True,
        # points='all',
        title="Distribution of Female Mortality Rates",
        labels={"mort_females": "Mortality Rate", "species": "Species"},
    )

    fig.update_layout(
        width=width,
        height=height,
        template="plotly_white",
        yaxis_tickformat=".0%",
        showlegend=False,
        font=dict(size=FONT_SIZE, family=FONT_FAMILY),
        title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY),
    )

    if transparent_bg:
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")

    return fig


def save_plot(fig, filename="violin_plot.png", scale=2):
    fig.write_image(filename, scale=scale)


violin_fig = create_violinplot(transparent_bg=True)
violin_fig.show()

# To save the image
save_plot(violin_fig, filename="bioassay_violin_plot.png", scale=3)

In [10]:
def create_stripplot(width=PLOT_WIDTH, height=PLOT_HEIGHT):
    fig = px.strip(
        df_clean,
        x='species',
        y='mort_females',
        color='species',
        color_discrete_map=TAXON_COLORS,
        title="Individual Mortality Measurements by Species",
        labels={
            'mort_females': 'Mortality Rate',
            'species': 'Species'
        }
    )
    
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    
    fig.update_layout(
        width=width,
        height=height,
        template="plotly_white",
        yaxis_tickformat='.0%',
        showlegend=False,
        font=dict(size=FONT_SIZE, family=FONT_FAMILY),
        title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY)
    )
    
    return fig

strip_fig = create_stripplot()
strip_fig.show()

In [11]:
def create_histogram(width=PLOT_WIDTH, height=PLOT_HEIGHT):
    fig = px.histogram(
        df_clean,
        x='mort_females',
        color='species',
        color_discrete_map=TAXON_COLORS,
        nbins=15,
        opacity=0.7,
        barmode='overlay',
        title="Distribution of Mortality Rates by Species",
        labels={
            'mort_females': 'Mortality Rate',
            'count': 'Frequency'
        }
    )
    
    fig.update_layout(
        width=width,
        height=height,
        template="plotly_white",
        xaxis_tickformat='.0%',
        font=dict(size=FONT_SIZE, family=FONT_FAMILY),
        title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY)
    )
    
    return fig

hist_fig = create_histogram()
hist_fig.show()

In [12]:
def create_manuscript_subplot(width=900, height=800):
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('A) Species Comparison', 'B) Distribution Analysis', 
                       'C) Individual Measurements', 'D) Summary Statistics'),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # A) Box plot data
    box_fig = px.box(df_clean, x='species', y='mort_females', color='species',
                     color_discrete_map=TAXON_COLORS, points='all')
    for trace in box_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=1, col=1)
    
    # B) Violin plot data
    violin_fig = px.violin(df_clean, x='species', y='mort_females', color='species',
                          color_discrete_map=TAXON_COLORS, box=True)
    for trace in violin_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=1, col=2)
    
    # C) Strip plot data
    strip_fig = px.strip(df_clean, x='species', y='mort_females', color='species',
                        color_discrete_map=TAXON_COLORS)
    for trace in strip_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=2, col=1)
    
    # D) Bar chart with summary stats
    summary = df_clean.groupby('species').agg({
        'mort_females': ['mean', 'std', 'count']
    }).round(3)
    summary.columns = ['mean', 'std', 'count']
    summary = summary.reset_index()
    summary['se'] = summary['std'] / np.sqrt(summary['count'])
    
    bar_fig = px.bar(summary, x='species', y='mean', color='species',
                     color_discrete_map=TAXON_COLORS, error_y='se')
    for trace in bar_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=2, col=2)
    
    # Add sample size annotations for the bar chart
    for i, (_, row) in enumerate(summary.iterrows()):
        fig.add_annotation(
            x=i,
            y=row['mean'] + row['se'] + 0.05,
            text=f"n={int(row['count'])}",
            showarrow=False,
            row=2, col=2,
            font=dict(size=FONT_SIZE-2, family=FONT_FAMILY)
        )
    
    # Update layout
    fig.update_layout(
        width=width,
        height=height,
        title_text="Mosquito Mortality Analysis",
        template="plotly_white",
        font=dict(size=FONT_SIZE, family=FONT_FAMILY),
        title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY)
    )
    
    # Update y-axes to show percentages
    for i in range(1, 3):
        for j in range(1, 3):
            fig.update_yaxes(tickformat='.0%', row=i, col=j)
    
    # Update axis titles
    fig.update_xaxes(title_text="Species", row=1, col=1, title_font=dict(size=FONT_SIZE, family=FONT_FAMILY))
    fig.update_xaxes(title_text="Species", row=1, col=2, title_font=dict(size=FONT_SIZE, family=FONT_FAMILY))
    fig.update_xaxes(title_text="Species", row=2, col=1, title_font=dict(size=FONT_SIZE, family=FONT_FAMILY))
    fig.update_xaxes(title_text="Species", row=2, col=2, title_font=dict(size=FONT_SIZE, family=FONT_FAMILY))
    
    fig.update_yaxes(title_text="Mortality Rate", row=1, col=1, title_font=dict(size=FONT_SIZE, family=FONT_FAMILY))
    fig.update_yaxes(title_text="Mortality Rate", row=1, col=2, title_font=dict(size=FONT_SIZE, family=FONT_FAMILY))
    fig.update_yaxes(title_text="Mortality Rate", row=2, col=1, title_font=dict(size=FONT_SIZE, family=FONT_FAMILY))
    fig.update_yaxes(title_text="Mean Mortality ± SE", row=2, col=2, title_font=dict(size=FONT_SIZE, family=FONT_FAMILY))
    
    return fig

manuscript_fig = create_manuscript_subplot()
manuscript_fig.show()